# 02 - XGBoost Model Optimization
This notebook trains an XGBoost classifier to detect chatter based on engineered features. 
We will drop specific features to prevent data leakage, use SMOTE for handling class imbalance, 
and utilize Optuna for Bayesian hyperparameter optimization targeting the PR-AUC score.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import optuna
import warnings

warnings.filterwarnings('ignore')

# Set plot styles for dark theme
plt.style.use('dark_background')

### 1. Data Loading and Feature Selection
We drop Fz_Std and Peak_FFT_Amp from our features as they were used directly to generate rule-based labels, which would cause severe data leakage.


In [ ]:
df = pd.read_csv('../data/processed/FINAL_ML_DATASET.csv')

print('Dataset Shape:', df.shape)
print('Class Distribution:\n', df['Label'].value_counts(normalize=True))

# Features and target
# Dropping features used for rule-based labelling to prevent data leakage
X = df.drop(columns=['Label', 'Fz_Std', 'Peak_FFT_Amp'])
y = df['Label']

print('\nFeatures used:', X.columns.tolist())

### 2. Stratified Train/Test Split and SMOTE
Our dataset contains approximately 17% positive (Chatter) instances. We first split the data with stratification to maintain this ratio in our test set.
Then, we apply SMOTE (Synthetic Minority Over-sampling Technique) exclusively to the training data to create synthetic positive samples, resulting in a balanced training set.


In [ ]:
# Stratified train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Before SMOTE, training class counts:')
print(y_train.value_counts())

# Apply SMOTE to the training set only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE, training class counts:')
print(y_train_sm.value_counts())

### 3. Hyperparameter Optimization with Optuna
We maximize the Precision-Recall Area Under Curve (PR-AUC), which is a better metric than ROC-AUC or Accuracy for imbalanced classification tasks.


In [ ]:
def objective(trial):
    # Define the hyperparameter search space
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'tree_method': 'hist',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        # Since data is balanced via SMOTE, scale_pos_weight is kept at 1
        'scale_pos_weight': 1 
    }
    
    # Train the XGBoost model
    model = xgb.XGBClassifier(**param)
    model.fit(X_train_sm, y_train_sm)
    
    # Predict probabilities for the positive class on the test set
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate PR-AUC
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = auc(recall, precision)
    
    return pr_auc

# Create an Optuna study to maximize PR-AUC
optuna.logging.set_verbosity(optuna.logging.WARNING) # Reduce noise
study = optuna.create_study(direction='maximize')

# Run optimization (reduced to 20 trials for demonstration, increase for better results)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('Best PR-AUC:', study.best_value)
print('Best hyperparameters:', study.best_params)

### 4. Final Model Evaluation
Train the final model using the optimal hyperparameters found by Optuna, and evaluate its performance.


In [ ]:
# Train the final model using the best hyperparameters
best_params = study.best_params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'aucpr'
best_params['tree_method'] = 'hist'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_sm, y_train_sm)

# Predictions
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

# 1. Classification Report
print('Classification Report:\n')
print(classification_report(y_test, y_pred))

# 2. Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Stable (0)', 'Chatter (1)'], 
            yticklabels=['Stable (0)', 'Chatter (1)'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# 3. Precision-Recall Curve Plot
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='#00E5A0', lw=2, label=f'XGBoost PR Curve (AUC = {pr_auc:.3f})')
# Calculate the baseline ratio of positive instances
baseline = y_test.sum() / len(y_test)
plt.axhline(y=baseline, color='white', linestyle='--', alpha=0.5, label=f'Baseline (AUC = {baseline:.3f})')
plt.title('Precision-Recall Curve for Chatter Detection')
plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision (PPV)')
plt.legend(loc='lower left')
plt.grid(alpha=0.2)
plt.show()